In [1]:
# --- Setup & download -----------------------------------------------------
# Hourly airport weather for ATL and DFW from the Iowa State Mesonet ASOS/METAR archive.
# Raw downloads are cached in data/weather/raw, so reruns do not hit the server again.
import io
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

STATIONS = ["ATL", "DFW"]                              # 3-letter IDs for US ASOS stations
START, END = "2023-01-01", "2025-08-01"                # END is exclusive; flights run to 2025-07-31
RWY_HEADING = {"ATL": 80, "DFW": 180}                  # degrees, from the Challenge 5 brief
AIRPORT_TZ = {"ATL": "America/New_York", "DFW": "America/Chicago"}

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "weather" / "raw"
OUT_DIR = DATA_DIR / "weather"
RAW_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"


def year_chunks(start, end):
    """Split [start, end) at calendar-year boundaries (one request per station-year)."""
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    edges = [s] + [pd.Timestamp(year=y, month=1, day=1) for y in range(s.year + 1, e.year + 1)] + [e]
    edges = sorted({x for x in edges if s <= x <= e})
    return [(a.strftime("%Y-%m-%d"), b.strftime("%Y-%m-%d")) for a, b in zip(edges[:-1], edges[1:])]


def fetch_chunk(station, sts, ets, retries=4):
    params = [
        ("station", station), ("data", "all"),
        ("sts", f"{sts}T00:00:00Z"), ("ets", f"{ets}T00:00:00Z"),
        ("tz", "UTC"), ("format", "onlycomma"),
        ("latlon", "no"), ("elev", "no"),
        ("missing", "M"), ("trace", "0.0001"),
        ("report_type", "3"), ("report_type", "4"),   # routine hourly + specials
    ]
    url = BASE_URL + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "acacia-hackathon-challenge5 (student project)"})
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=180) as resp:
                text = resp.read().decode("utf-8", errors="replace")
            if not text.lstrip().startswith(("station,valid", "#")):
                raise ValueError("unexpected response: " + text[:200])
            return text
        except (urllib.error.URLError, TimeoutError, ValueError) as err:   # includes 429 / 503
            print(f"  attempt {attempt} failed: {err}")
            if attempt == retries:
                raise
            time.sleep(10 * attempt)


def download_asos():
    paths = []
    for st in STATIONS:
        for sts, ets in year_chunks(START, END):
            path = RAW_DIR / f"asos_{st}_{sts}_{ets}.csv"
            if not path.exists():
                print(f"downloading {st} {sts} -> {ets}")
                path.write_text(fetch_chunk(st, sts, ets), encoding="utf-8")
                time.sleep(2)                          # be polite: the server rate-limits by IP
            paths.append(path)
    return paths


raw_files = download_asos()
raw = pd.concat(
    [pd.read_csv(p, na_values=["M", ""], comment="#", low_memory=False) for p in raw_files],
    ignore_index=True,
)
print("raw observations:", raw.shape)
raw.head(3)

downloading ATL 2023-01-01 -> 2024-01-01
downloading ATL 2024-01-01 -> 2025-01-01
downloading ATL 2025-01-01 -> 2025-08-01
downloading DFW 2023-01-01 -> 2024-01-01
downloading DFW 2024-01-01 -> 2025-01-01
downloading DFW 2025-01-01 -> 2025-08-01
raw observations: (52843, 30)


,station,valid,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,vsby,gust,skyc1,skyc2,skyc3,skyc4,skyl1,skyl2,skyl3,skyl4,wxcodes,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_drct,peak_wind_time,feel,metar,snowdepth
0,ATL,2023-01-01 00:52,58.0,56.0,93.04,0.0,0.0,0.0,30.03,1016.7,10.0,NaN,FEW,NaN,NaN,NaN,25000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.9,KATL 010052Z 00000KT 10SM FEW250 14/13 A3003 R...,NaN
1,ATL,2023-01-01 01:52,57.0,55.0,93.01,210.0,5.0,0.0,30.04,1016.9,10.0,NaN,CLR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.0,KATL 010152Z 21005KT 10SM CLR 14/13 A3004 RMK ...,NaN
2,ATL,2023-01-01 02:52,56.0,55.0,96.78,230.0,4.0,0.0,30.04,1017.3,10.0,NaN,FEW,NaN,NaN,NaN,300.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55.9,KATL 010252Z COR 23004KT 10SM R09R/5500VP6000F...,NaN


# Challenge 5 - hourly weather from the Mesonet METAR archive

**Why not aviationweather.gov?** Its Data API only keeps about 30 days of history. The Iowa State Mesonet archive covers 2023 to 2025 and, unlike the Meteostat columns in the flight file, includes **wind direction**, **gusts**, **visibility**, **cloud ceiling** and **present weather**.

This notebook:
1. cleans and converts the raw observations,
2. computes crosswind and tailwind from the runway headings in the brief,
3. aggregates to one row per airport per hour (with explicit gaps),
4. checks the result against the old Meteostat weather,
5. shows how to attach it to flights with a 3 to 12 hour lead and no leakage.

## 1. Clean and convert

Units: temperature in °C, wind in knots, visibility in statute miles, pressure in hPa, precipitation in mm. Ceiling is the lowest broken, overcast or vertical-visibility layer.

In [2]:
def clean_raw(raw):
    df = raw.copy()
    df = df.rename(columns={"station": "airport"})
    df["valid_utc"] = pd.to_datetime(df["valid"], utc=True)

    num_cols = ["tmpf", "dwpf", "relh", "drct", "sknt", "p01i", "alti", "mslp", "vsby", "gust",
                "skyl1", "skyl2", "skyl3", "skyl4"]
    txt_cols = ["skyc1", "skyc2", "skyc3", "skyc4", "wxcodes"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce") if c in df.columns else np.nan
    for c in txt_cols:
        df[c] = df[c].astype("string").str.upper() if c in df.columns else pd.Series(pd.NA, index=df.index, dtype="string")

    out = pd.DataFrame({
        "airport": df["airport"],
        "valid_utc": df["valid_utc"],
        "temp_c": (df["tmpf"] - 32) * 5 / 9,
        "dewpt_c": (df["dwpf"] - 32) * 5 / 9,
        "relh": df["relh"],
        "slp_hpa": df["mslp"],
        "wind_dir_deg": df["drct"],              # direction the wind blows FROM, true north
        "wind_kt": df["sknt"],
        "gust_kt": df["gust"],
        "vis_mi": df["vsby"],
        "precip_mm": df["p01i"] * 25.4,
        "wxcodes": df["wxcodes"],
    })

    layers = [df[f"skyl{i}"].where(df[f"skyc{i}"].isin(["BKN", "OVC", "VV"])) for i in range(1, 5)]
    out["ceiling_ft"] = pd.concat(layers, axis=1).min(axis=1)

    wx = out["wxcodes"].fillna("")
    out["thunder"] = wx.str.contains("TS")
    out["precip_type"] = wx.str.contains(r"RA|SN|DZ|GR|GS|PL|SG|IC|UP")
    return out.sort_values(["airport", "valid_utc"]).drop_duplicates(["airport", "valid_utc"]).reset_index(drop=True)


obs = clean_raw(raw)
print(obs.shape, "| airports:", obs["airport"].unique().tolist())
print("share missing:\n", obs[["temp_c", "wind_dir_deg", "wind_kt", "gust_kt", "vis_mi", "slp_hpa"]].isna().mean().round(3))
obs.head()

(52841, 15) | airports: ['ATL', 'DFW']
share missing:
 temp_c          0.000
wind_dir_deg    0.016
wind_kt         0.000
gust_kt         0.834
vis_mi          0.000
slp_hpa         0.145
dtype: float64


,airport,valid_utc,temp_c,dewpt_c,relh,slp_hpa,wind_dir_deg,wind_kt,gust_kt,vis_mi,precip_mm,wxcodes,ceiling_ft,thunder,precip_type
0,ATL,2023-01-01 00:52:00+00:00,14.444444,13.333333,93.04,1016.7,0.0,0.0,NaN,10.0,0.0,<NA>,NaN,False,False
1,ATL,2023-01-01 01:52:00+00:00,13.888889,12.777778,93.01,1016.9,210.0,5.0,NaN,10.0,0.0,<NA>,NaN,False,False
2,ATL,2023-01-01 02:52:00+00:00,13.333333,12.777778,96.78,1017.3,230.0,4.0,NaN,10.0,0.0,<NA>,NaN,False,False
3,ATL,2023-01-01 03:52:00+00:00,13.333333,12.777778,96.78,1017.4,250.0,3.0,NaN,6.0,0.0,BR,NaN,False,False
4,ATL,2023-01-01 04:38:00+00:00,13.333333,12.777778,96.78,NaN,240.0,7.0,NaN,0.0,0.0,FG,NaN,False,False


## 2. Crosswind and tailwind

The brief says every ATL flight uses heading $\theta_{rwy} = 80^\circ$ and every DFW flight uses $180^\circ$. With wind speed $V$ and direction $\theta_w$ (the direction the wind comes from):

$$V_{head} = V \cos(\theta_w - \theta_{rwy}), \qquad V_{cross} = V \sin(\theta_w - \theta_{rwy})$$

$V_{head} > 0$ is a headwind and $V_{head} < 0$ is a **tailwind**. $|V_{cross}|$ is the crosswind. I compute this for the sustained wind and for the peak wind $V_{peak} = \max(V, V_{gust})$ (same direction). A calm wind gives zero components. Variable wind with no direction gives missing components.

In [3]:
def wind_components(speed, direction, heading):
    ang = np.deg2rad(direction - heading)
    head = np.where(speed == 0, 0.0, speed * np.cos(ang))
    cross = np.where(speed == 0, 0.0, speed * np.sin(ang))
    return head, cross


heading = obs["airport"].map(RWY_HEADING)
peak = np.fmax(obs["wind_kt"], obs["gust_kt"])

obs["headwind_kt"], obs["crosswind_kt"] = wind_components(obs["wind_kt"], obs["wind_dir_deg"], heading)
obs["head_peak_kt"], obs["cross_peak_kt"] = wind_components(peak, obs["wind_dir_deg"], heading)
obs["peak_kt"] = peak

print("components missing (variable wind, no direction):", int(obs["headwind_kt"].isna().sum()))
obs[["airport", "valid_utc", "wind_dir_deg", "wind_kt", "gust_kt", "headwind_kt", "crosswind_kt"]].dropna().sample(5, random_state=0)

components missing (variable wind, no direction): 861


,airport,valid_utc,wind_dir_deg,wind_kt,gust_kt,headwind_kt,crosswind_kt
38402,DFW,2024-03-03 19:53:00+00:00,180.0,15.0,22.0,15.000000,0.000000
38429,DFW,2024-03-04 17:53:00+00:00,210.0,16.0,22.0,13.856406,8.000000
17843,ATL,2024-09-27 06:52:00+00:00,50.0,17.0,24.0,14.722432,-8.500000
20629,ATL,2025-01-07 07:25:00+00:00,340.0,14.0,28.0,-2.431074,-13.787309
27729,DFW,2023-02-12 21:53:00+00:00,210.0,12.0,18.0,10.392305,6.000000


## 3. Aggregate to one row per airport per hour

Hour buckets are in **UTC** (no daylight-saving ambiguity). Within each hour:

- state variables come from the **last** observation (`temp_c`, `wind_kt`, `vis_mi`, `headwind_kt`, ...);
- worst-case variables use the whole hour: `peak_wind_kt`, `xwind_max_kt`, `tailwind_max_kt`, `vis_min_mi`, `ceiling_min_ft`, `precip_mm` (max, because `p01i` is a running one-hour total), `thunder`;
- `known_at_utc` is the time of the last observation used. **Any feature for hour $h$ must only be used for events after `known_at_utc`.**

The result is reindexed onto the full hourly grid, so hours with no observation show up as `n_obs = 0` instead of silently disappearing. `flight_category` follows the US definitions (LIFR, IFR, MVFR, VFR) using the last observation's visibility and ceiling.

In [4]:
LAST_COLS = ["temp_c", "dewpt_c", "relh", "slp_hpa", "wind_dir_deg", "wind_kt", "gust_kt",
             "vis_mi", "ceiling_ft", "headwind_kt", "crosswind_kt", "wxcodes"]

obs["hour_utc"] = obs["valid_utc"].dt.floor("h")
obs["abs_cross_peak"] = obs["cross_peak_kt"].abs()
obs["tail_peak"] = (-obs["head_peak_kt"]).clip(lower=0)

g = obs.groupby(["airport", "hour_utc"])
last = g.tail(1).set_index(["airport", "hour_utc"])[LAST_COLS + ["valid_utc"]].rename(columns={"valid_utc": "known_at_utc"})
agg = g.agg(
    n_obs=("valid_utc", "size"),
    peak_wind_kt=("peak_kt", "max"),
    xwind_max_kt=("abs_cross_peak", "max"),
    tailwind_max_kt=("tail_peak", "max"),
    vis_min_mi=("vis_mi", "min"),
    ceiling_min_ft=("ceiling_ft", "min"),
    precip_mm=("precip_mm", "max"),
    thunder=("thunder", "max"),
    precip_type=("precip_type", "max"),
)
hourly = last.join(agg)

# full hourly grid per airport, so gaps are explicit
first_hour = pd.Timestamp(START, tz="UTC")
last_hour = pd.Timestamp(END, tz="UTC") - pd.Timedelta(hours=1)
grid = pd.MultiIndex.from_product(
    [STATIONS, pd.date_range(first_hour, last_hour, freq="h")], names=["airport", "hour_utc"]
)
hourly = hourly.reindex(grid)
hourly["n_obs"] = hourly["n_obs"].fillna(0).astype(int)
hourly[["thunder", "precip_type"]] = hourly[["thunder", "precip_type"]].astype("boolean")
hourly = hourly.reset_index()

# US flight categories from the last observation (no ceiling = unlimited, missing visibility = 10 mi)
ceil = hourly["ceiling_ft"].fillna(np.inf)
vis = hourly["vis_mi"].fillna(10.0)
hourly["flight_category"] = np.select(
    [(ceil < 500) | (vis < 1), (ceil < 1000) | (vis < 3), (ceil <= 3000) | (vis <= 5)],
    ["LIFR", "IFR", "MVFR"], default="VFR",
)
hourly.loc[hourly["n_obs"] == 0, "flight_category"] = pd.NA

# local clock time (naive) for readability; repeats once a year at the DST fall-back hour
hourly["time_local"] = pd.NaT
for ap, tz in AIRPORT_TZ.items():
    m = hourly["airport"] == ap
    hourly.loc[m, "time_local"] = hourly.loc[m, "hour_utc"].dt.tz_convert(tz).dt.tz_localize(None)

hourly.to_csv(OUT_DIR / "hourly_atl_dfw.csv", index=False)
print(hourly.shape)
hourly.groupby("airport").agg(
    hours=("hour_utc", "size"),
    with_obs=("n_obs", lambda s: (s > 0).sum()),
    mean_obs_per_hour=("n_obs", "mean"),
)

(45264, 26)


,hours,with_obs,mean_obs_per_hour
airport,,,
ATL,22632,22620,1.168964
DFW,22632,22627,1.165827


In [5]:
hourly.dropna(subset=["temp_c"]).head()

,airport,hour_utc,temp_c,dewpt_c,relh,slp_hpa,wind_dir_deg,wind_kt,gust_kt,vis_mi,ceiling_ft,headwind_kt,crosswind_kt,wxcodes,known_at_utc,n_obs,peak_wind_kt,xwind_max_kt,tailwind_max_kt,vis_min_mi,ceiling_min_ft,precip_mm,thunder,precip_type,flight_category,time_local
0,ATL,2023-01-01 00:00:00+00:00,14.444444,13.333333,93.04,1016.7,0.0,0.0,NaN,10.0,NaN,0.000000,0.000000,<NA>,2023-01-01 00:52:00+00:00,1,0.0,0.000000,-0.000000,10.0,NaN,0.0,False,False,VFR,2022-12-31 19:00:00
1,ATL,2023-01-01 01:00:00+00:00,13.888889,12.777778,93.01,1016.9,210.0,5.0,NaN,10.0,NaN,-3.213938,3.830222,<NA>,2023-01-01 01:52:00+00:00,1,5.0,3.830222,3.213938,10.0,NaN,0.0,False,False,VFR,2022-12-31 20:00:00
2,ATL,2023-01-01 02:00:00+00:00,13.333333,12.777778,96.78,1017.3,230.0,4.0,NaN,10.0,NaN,-3.464102,2.000000,<NA>,2023-01-01 02:52:00+00:00,1,4.0,2.000000,3.464102,10.0,NaN,0.0,False,False,VFR,2022-12-31 21:00:00
3,ATL,2023-01-01 03:00:00+00:00,13.333333,12.777778,96.78,1017.4,250.0,3.0,NaN,6.0,NaN,-2.954423,0.520945,BR,2023-01-01 03:52:00+00:00,1,3.0,0.520945,2.954423,6.0,NaN,0.0,False,False,VFR,2022-12-31 22:00:00
4,ATL,2023-01-01 04:00:00+00:00,13.333333,12.777778,96.78,1017.2,230.0,7.0,NaN,0.0,NaN,-6.062178,3.500000,FG,2023-01-01 04:52:00+00:00,3,7.0,3.500000,6.577848,0.0,NaN,0.0,False,False,LIFR,2022-12-31 23:00:00


## 4. Quality checks

Compare against the Meteostat weather already in the flight file (`data/clean/weather_clean.csv`, local scheduled hour, wind in km/h). The two sources should agree closely for temperature and pressure. Wind should agree after converting km/h to knots ($1\ \text{km/h} = 0.54\ \text{kt}$).

In [6]:
print("hours without any observation:")
print(hourly.groupby("airport")["n_obs"].apply(lambda s: (s == 0).sum()))
print("\nflight category share (hours with data):")
print(hourly.dropna(subset=["flight_category"]).groupby("airport")["flight_category"].value_counts(normalize=True).round(3))
print("\nmax crosswind / tailwind seen (kt):")
print(hourly.groupby("airport")[["xwind_max_kt", "tailwind_max_kt", "peak_wind_kt"]].max().round(1))

old_path = DATA_DIR / "clean" / "weather_clean.csv"
if old_path.exists():
    old = pd.read_csv(old_path, parse_dates=["time"])
    new = hourly.dropna(subset=["temp_c"]).drop_duplicates(["airport", "time_local"])
    cmp = old.merge(new, left_on=["airport", "time"], right_on=["airport", "time_local"], how="inner")
    print(f"\nmatched {len(cmp)} of {len(old)} old weather rows")
    print("mean |temp diff| (C):     ", round((cmp["temp"] - cmp["temp_c"]).abs().mean(), 2))
    print("mean |pressure diff| (hPa):", round((cmp["pres"] - cmp["slp_hpa"]).abs().mean(), 2))
    print("mean |wind diff| (kt):     ", round((cmp["wspd"] * 0.5400 - cmp["wind_kt"]).abs().mean(), 2))
else:
    print("\n(data/clean/weather_clean.csv not found - run data_cleaning.ipynb to enable this comparison)")

hours without any observation:
airport
ATL    12
DFW     5
Name: n_obs, dtype: int64

flight category share (hours with data):
airport  flight_category
ATL      VFR                0.835
         MVFR               0.085
         IFR                0.051
         LIFR               0.029
DFW      VFR                0.850
         MVFR               0.102
         IFR                0.034
         LIFR               0.014
Name: proportion, dtype: float64

max crosswind / tailwind seen (kt):
         xwind_max_kt  tailwind_max_kt  peak_wind_kt
airport                                             
ATL              58.0             59.1          62.0
DFW              63.0             66.1          67.0

(data/clean/weather_clean.csv not found - run data_cleaning.ipynb to enable this comparison)


## 5. Attach weather to flights with an exact prediction cutoff

`attach_weather` is imported from `stage1_pipeline.py`, the shared Stage 1 pipeline. It converts the scheduled endpoint to UTC and selects the most recent hourly summary whose last contributing observation is at or before `event_utc - lead_hours`. Summaries older than two hours are unavailable.

Unresolved daylight-saving times remain missing and their flight IDs are retained. Observation timestamps proxy publication time; actual ingestion latency is not reconstructed.

Use `sched_arr` for destination events. The old `arr_wx_hour` identifies the original file’s weather join, which can be the previous day. Run `python stage1_pipeline.py` for TAF forecasts and BTS context.


In [ ]:
from stage1_pipeline import attach_weather

flights_path = DATA_DIR / "clean" / "flights_clean.csv"
if flights_path.exists():
    flights = pd.read_csv(flights_path, parse_dates=["sched_dep", "sched_arr"])
    for lead in (3, 6, 12):
        w = attach_weather(flights, hourly, "ORIGIN", lead)
        have = w[f"origin_temp_c_lag{lead}h"].notna().mean()
        print(f"origin weather {lead:2d}h ahead: {len(w)} flights, {have:.1%} have an observation")
    demo = attach_weather(flights, hourly, "ORIGIN", 3)
    demo.head()
else:
    print("data/clean/flights_clean.csv not found - run data_cleaning.ipynb first")